You are analyzing a social network dataset at Google. Your task is to find mutual friends between two users, Karl and Hans. There is only one user named Karl and one named Hans in the dataset.

The output should contain 'user_id' and 'user_name' columns.

🔗 Understanding how to join tables in SQL is essential for effective data analysis; mastering this concept allows you to combine related data seamlessly. Give it a try! 👇

In [0]:
%skip
%sql
CREATE TABLE ska_catalog.bronze.google_users(user_id INT, user_name varchar(30));
INSERT INTO ska_catalog.bronze.google_users VALUES (1, 'Karl'), (2, 'Hans'), (3, 'Emma'), (4, 'Emma'), (5, 'Mike'), (6, 'Lucas'), (7, 'Sarah'), (8, 'Lucas'), (9, 'Anna'), (10, 'John');

CREATE TABLE ska_catalog.bronze.google_friends(user_id INT, friend_id INT);
INSERT INTO ska_catalog.bronze.google_friends VALUES (1,3),(1,5),(2,3),(2,4),(3,1),(3,2),(3,6),(4,7),(5,8),(6,9),(7,10),(8,6),(9,10),(10,7),(10,9);

In [0]:
WITH karl_friends AS (
  SELECT friend_id 
  FROM ska_catalog.bronze.google_friends
  WHERE user_id = (SELECT user_id FROM ska_catalog.bronze.google_users WHERE user_name = "Karl")
),
hans_friends AS (
  SELECT friend_id
  FROM ska_catalog.bronze.google_friends
  WHERE user_id = (SELECT user_id FROM ska_catalog.bronze.google_users WHERE user_name = "Hans")
)
SELECT user_id,user_name
FROM ska_catalog.bronze.google_users U
JOIN karl_friends k ON k.friend_id = U.user_id
JOIN hans_friends h ON h.friend_id = U.user_id

In [0]:
-- List all users who have more than 2 friends.
SELECT user_id, Name, followers_count
FROM (SELECT
*,
DENSE_RANK() OVER (ORDER BY followers_count DESC) AS rank
FROM (
    SELECT 
      u.user_id, 
      u.user_name AS `Name`,
      count( f.friend_id) AS `followers_count`
    FROM ska_catalog.bronze.google_users u
    FULL OUTER JOIN ska_catalog.bronze.google_friends f
    ON f.user_id = u.user_id
    GROUP BY Name, u.user_id
  )
  ORDER BY rank
)
WHERE followers_count >= 2
ORDER BY  followers_count DESC, user_id ASC ;

In [0]:
SELECT * FROM ska_catalog.bronze.google_users

In [0]:
-- Find users who are friends with 'Emma'.
SELECT user_name AS `Friends of Emma` FROM ska_catalog.bronze.google_users
WHERE user_id IN
(SELECT friend_id FROM ska_catalog.bronze.google_friends
WHERE user_id IN (SELECT DISTINCT user_id FROM ska_catalog.bronze.google_users WHERE user_name = 'Emma')
)

In [0]:
-- Find users who are friends with 'Emma' (i.e., users that Emma has added as a friend).
SELECT user_name AS `Friends of Emma` FROM ska_catalog.bronze.google_users
WHERE user_id IN
(SELECT friend_id FROM ska_catalog.bronze.google_friends
WHERE user_id IN (SELECT DISTINCT user_id FROM ska_catalog.bronze.google_users WHERE user_name = 'Emma')
)

In [0]:
-- Get the number of mutual friendships (i.e., both users are friends with each other).
WITH join_users_fnds AS (
  SELECT u.user_id, u.user_name, f.friend_id
  FROM ska_catalog.bronze.google_users u
  JOIN ska_catalog.bronze.google_friends f
  ON u.user_id = f.user_id
)
SELECT concat(j.user_name,' follows ', j2.user_name, ' and ',  j2.user_name, ' follows ', j.user_name ) AS DSC
FROM join_users_fnds j
JOIN join_users_fnds j2
ON j2.user_id = j.friend_id AND j2.friend_id = j.user_id
-- This condition ensures each mutual friendship pair is counted only once, 
-- avoiding double-counting since (A,B) and (B,A) are both present for mutual friendships.
WHERE j.user_id < j.friend_id

In [0]:
-- Find users who have no friends.
SELECT user_id, user_name 
FROM ska_catalog.bronze.google_users
WHERE user_id NOT IN (SELECT DISTINCT user_id FROM ska_catalog.bronze.google_friends)

In [0]:
-- List each user and the number of times they appear as a friend.
SELECT u.user_id, u.user_name, COUNT(f.friend_id) AS times_as_friend
FROM ska_catalog.bronze.google_users u
LEFT JOIN ska_catalog.bronze.google_friends f
ON u.user_id = f.friend_id
GROUP BY u.user_id, u.user_name
ORDER BY times_as_friend DESC, u.user_id ASC

In [0]:
-- Find the top 3 users with the most friends.
SELECT u.user_id,u.user_name, COUNT(f.friend_id) AS `fnd_cnt` FROM ska_catalog.bronze.google_users u
JOIN ska_catalog.bronze.google_friends f
ON u.user_id = f.user_id
GROUP BY u.user_id, u.user_name
ORDER BY fnd_cnt DESC
LIMIT 3;

In [0]:
-- Find users who share the same friend (i.e., friend_id is common between two different users).

WITH shared_friends AS (
  SELECT friend_id
  FROM ska_catalog.bronze.google_friends
  GROUP BY friend_id
  HAVING COUNT(DISTINCT user_id) > 1
)
SELECT u1.user_name AS user1, u2.user_name AS user2, sf.friend_id
FROM ska_catalog.bronze.google_friends f1
JOIN ska_catalog.bronze.google_friends f2
  ON f1.friend_id = f2.friend_id AND f1.user_id < f2.user_id
JOIN ska_catalog.bronze.google_users u1 ON f1.user_id = u1.user_id
JOIN ska_catalog.bronze.google_users u2 ON f2.user_id = u2.user_id
JOIN shared_friends sf ON f1.friend_id = sf.friend_id
ORDER BY sf.friend_id, user1, user2

In [0]:
-- Using a CTE, find users who are friends with someone named 'Lucas' and count how many such users exist.
WITH lucas_users AS (
  SELECT user_id
  FROM ska_catalog.bronze.google_users
  WHERE user_name = 'Lucas'
),
friends_of_lucas AS (
  SELECT user_id
  FROM ska_catalog.bronze.google_friends
  WHERE friend_id IN (SELECT user_id FROM lucas_users)
)
SELECT COUNT(DISTINCT user_id) AS num_friends_of_lucas
FROM friends_of_lucas;

In [0]:
-- Find users who are friends with the same person more than once (duplicate friendships).
SELECT user_id, friend_id, COUNT(*) AS duplicate_count
FROM ska_catalog.bronze.google_friends
GROUP BY user_id, friend_id
HAVING COUNT(*) > 1
ORDER BY user_id, friend_id

Find all records from days when the number of distinct users receiving emails was greater than the number of distinct users sending emails.

In [0]:
%skip
CREATE TABLE ska_catalog.bronze.google_gmail_emails (id INT, from_user VARCHAR(50), to_user VARCHAR(50), day INT);

INSERT INTO ska_catalog.bronze.google_gmail_emails (id, from_user, to_user, day) VALUES(0, '6edf0be4b2267df1fa', '75d295377a46f83236', 10),(1, '6edf0be4b2267df1fa', '32ded68d89443e808', 6),(2, '6edf0be4b2267df1fa', '55e60cfcc9dc49c17e', 10),(3, '6edf0be4b2267df1fa', 'e0e0defbb9ec47f6f7', 6),(4, '6edf0be4b2267df1fa', '47be2887786891367e', 1),(5, '6edf0be4b2267df1fa', '2813e59cf6c1ff698e', 6),(6, '6edf0be4b2267df1fa', 'a84065b7933ad01019', 8),(7, '6edf0be4b2267df1fa', '850badf89ed8f06854', 1),(8, '6edf0be4b2267df1fa', '6b503743a13d778200', 1),(9, '6edf0be4b2267df1fa', 'd63386c884aeb9f71d', 3),(10, '6edf0be4b2267df1fa', '5b8754928306a18b68', 2),(11, '6edf0be4b2267df1fa', '6edf0be4b2267df1fa', 8),(12, '6edf0be4b2267df1fa', '406539987dd9b679c0', 9),(13, '6edf0be4b2267df1fa', '114bafadff2d882864', 5),(14, '6edf0be4b2267df1fa', '157e3e9278e32aba3e', 2),(15, '75d295377a46f83236', '75d295377a46f83236', 6),(16, '75d295377a46f83236', 'd63386c884aeb9f71d', 8),(17, '75d295377a46f83236', '55e60cfcc9dc49c17e', 3),(18, '75d295377a46f83236', '47be2887786891367e', 10),(19, '75d295377a46f83236', '5b8754928306a18b68', 10),(20, '75d295377a46f83236', '850badf89ed8f06854', 7);

In [0]:
SELECT * FROM ska_catalog.bronze.google_gmail_emails

In [0]:
-- Find all records from days when the number of distinct users receiving emails was greater than the number of distinct users sending emails.
WITH dist_user AS (
  SELECT day ,
    COUNT(DISTINCT from_user) AS distinct_from_users,
    COUNT(DISTINCT to_user) AS distinct_to_user
  FROM ska_catalog.bronze.google_gmail_emails
  GROUP BY day
)
SELECT a.id,a.from_user,a.to_user,a.day FROM ska_catalog.bronze.google_gmail_emails a
JOIN dist_user ON dist_user.day = a.day
WHERE dist_user.distinct_to_user > dist_user.distinct_from_users

In [0]:
SELECT day ,
    COUNT(DISTINCT from_user) AS distinct_from_users,
    COUNT(DISTINCT to_user) AS distinct_to_user
  FROM ska_catalog.bronze.google_gmail_emails
  GROUP BY day